# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Veduisback/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [19]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF token loaded:", HF_TOKEN is not None)

HF token loaded: True


In [20]:
import duckdb

con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute("INSTALL azure")
con.execute("LOAD azure")

print("DuckDB ready.")

DuckDB ready.


In [21]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
""")

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


In [22]:
import requests

url = "https://huggingface.co/api/datasets/FlyRank/internship-warehouse/tree/main"
headers = {"Authorization": f"Bearer {HF_TOKEN}"}

response = requests.get(url, headers=headers)

print("Status:", response.status_code)
print(response.text[:5000])

Status: 200
[{"type":"directory","oid":"a9d0e87db2923894c18291294bd3818f00a5791e","size":0,"path":"fact_content_daily_performance"},{"type":"file","oid":"bed0738c7eeb449bca98b5d2f33c89a1ee56349a","size":2504,"path":".gitattributes"},{"type":"file","oid":"af501786cb97002e9646649c488d8f6e4f275fab","size":3038,"path":"README.md"},{"type":"file","oid":"7c5ee024976718db679c472b80e9d44114b83dfd","size":3377,"lfs":{"oid":"b64a9f13c102b7563f1f632c20a8680cb4a9b2fa4def0f91632128fb47d9537c","size":3377,"pointerSize":129},"xetHash":"a814f45c06fab5b9b2c1919c017c47c86637441af390e1acf6cabd38d133efe5","path":"dim_clients.parquet"},{"type":"file","oid":"0293c81e1cd756c034aeab1420a7d5fca2d5e347","size":19557889,"lfs":{"oid":"92978737e356996b4bb23e9f207457efed84b8c0a45486e8a66a56163cba1341","size":19557889,"pointerSize":133},"xetHash":"f1cedbef336e6849af87659d628440a3c23bfb7a2c335baaf97ce3a4ac245bfd","path":"dim_content.parquet"},{"type":"file","oid":"3a1435ed84d54ab479adc47836f6fb29dbc58f17","size":1450

In [23]:
MARCH_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

march_schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{MARCH_PATH}')"
).df()

display(march_schema)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [24]:
march_preview = con.execute(
    f"SELECT * FROM read_parquet('{MARCH_PATH}') LIMIT 5"
).df()

display(march_preview)

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [25]:
content_schema = con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    )
""").df()

display(content_schema)

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [26]:
content_preview = con.execute("""
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    )
    LIMIT 5
""").df()

display(content_preview)


,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682,2555,NaT,NaT,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438,2430,NaT,NaT,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576,2645,NaT,NaT,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457,2522,NaT,NaT,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776,2552,NaT,NaT,True,False


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### 1. Row grain

One row represents one page/content item for one client on one reporting date in the daily performance fact table.

For the modeling feature frame, I will aggregate these daily observations to the content/client level for the decision window.

### 2. Tables

I will use `fact_content_daily_performance` for daily Google Search Console and Google Analytics measurements, and `dim_content` for page/content attributes such as search volume, content type, word count, and content dates.

### 3. Time window

I will use February 2026 as the feature window and March 2026 as the subsequent outcome window.

This keeps the decision moment before the outcome is observed. June 2026 is treated as a sealed final month and is not used for feature or label development.

### 4. What I predict/rank

The lane ranks pages by priority for human content review.

For this exercise, the future outcome is represented by an observed proxy rather than a claim about guaranteed future performance.

### 5. Deliberate exclusion

I deliberately exclude information from the future outcome window and any label-derived field from the feature set.

In particular, March performance cannot be used as a February decision-time feature because that would leak future information into the model.

In [27]:
q1 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id) AS distinct_clients,
    COUNT(DISTINCT content_hash_id) AS distinct_content_items,
    COUNT(DISTINCT CONCAT(
        client_hash_id, '||', content_hash_id, '||', CAST(report_date AS VARCHAR)
    )) AS distinct_client_content_dates
FROM read_parquet('{MARCH_PATH}')
"""

display(con.execute(q1).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_clients,distinct_content_items,distinct_client_content_dates
0,9841378,55,331437,9841378


### Verification 1 — grain

The March 2026 fact slice contains 9,841,378 rows.

The `client_hash_id + content_hash_id + report_date` combination is unique across the slice because its distinct count is also 9,841,378.

This verifies that one fact row represents one client × content item × reporting-date observation.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [28]:
q2 = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet('{MARCH_PATH}')
"""

display(con.execute(q2).df())

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


### Verification 2 — slice size and date span

The March 2026 slice contains 9,841,378 rows and spans from 2026-03-01 through 2026-03-31.

This confirms that the selected warehouse partition covers the expected full March reporting window.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [29]:
q3 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,
    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows
FROM read_parquet('{MARCH_PATH}')
"""

display(con.execute(q3).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


### Verification 3 — data availability

For March 2026, the warehouse contains 9,841,378 total fact rows.

Using the explicit `IS TRUE` availability flags:
- GSC data is available for 3,611,061 rows.
- GA4 data is available for 413,966 rows.

This verifies that data availability must be determined from the warehouse availability flags rather than inferred from metric values being non-null or non-zero.

In [30]:
FEB_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-02/*.parquet"
)

feature_sql = f"""
WITH daily AS (
    SELECT
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        sessions_organic
    FROM read_parquet('{FEB_PATH}')
    WHERE gsc_data_available IS TRUE
),

performance AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions_30d,

        SUM(gsc_clicks) AS clicks_30d,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr_30d,

        AVG(gsc_avg_position) AS avg_position_30d,

        SUM(sessions_organic) AS organic_sessions_30d

    FROM daily
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM performance
"""

features_df = con.execute(feature_sql).df()

print("Feature frame shape:", features_df.shape)
display(features_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (153559, 7)


,client_hash_id,content_hash_id,impressions_30d,clicks_30d,ctr_30d,avg_position_30d,organic_sessions_30d
0,client_e547b89c05043229,content_7995404695ee1ffd,1012.0,3.0,0.002964,29.609070,5.0
1,client_e547b89c05043229,content_ccbb253f142217c3,1598.0,7.0,0.004380,17.806923,5.0
2,client_e547b89c05043229,content_ae16a6b9cf64c80a,861.0,0.0,0.000000,7.852945,0.0
3,client_e547b89c05043229,content_acf700633f016e5a,245.0,0.0,0.000000,7.123694,0.0
4,client_e547b89c05043229,content_712e44562fed8ff2,306.0,0.0,0.000000,7.965842,0.0


### Feature availability at the decision moment

The decision moment is the end of the February 2026 feature window, before the March outcome window.

| Feature | Why it is knowable at the decision moment |
|---|---|
| `impressions_30d` | It is calculated from GSC impressions already observed during the February 30-day window. |
| `clicks_30d` | It is calculated from GSC clicks already observed during the February 30-day window. |
| `ctr_30d` | It is calculated from February clicks and impressions, both already observed before the decision. |
| `avg_position_30d` | It is calculated from GSC average-position observations recorded during February. |
| `organic_sessions_30d` | It is calculated from organic sessions already observed during the February window. |

No March performance values are used in these five features, so the feature frame represents information available before the March outcome window.

In [31]:
MARCH_OUTCOME_SQL = f"""
SELECT
    client_hash_id,
    content_hash_id,
    CASE
        WHEN SUM(gsc_impressions) > 0 THEN 1
        ELSE 0
    END AS march_had_impressions
FROM read_parquet('{MARCH_PATH}')
WHERE gsc_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
"""

outcome_df = con.execute(MARCH_OUTCOME_SQL).df()

print("Outcome frame shape:", outcome_df.shape)
display(outcome_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Outcome frame shape: (176738, 3)


,client_hash_id,content_hash_id,march_had_impressions
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,1
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,1
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,1
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,1
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,1


In [32]:
leakage_df = features_df.merge(
    outcome_df,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Merged rows:", len(leakage_df))
print("Outcome distribution:")
display(leakage_df["march_had_impressions"].value_counts())

Merged rows: 134238
Outcome distribution:


,count
march_had_impressions,
1,134238


In [33]:
MARCH_OUTCOME_SQL = f"""
SELECT
    client_hash_id,
    content_hash_id,
    CASE
        WHEN SUM(COALESCE(gsc_impressions, 0)) > 0 THEN 1
        ELSE 0
    END AS march_had_impressions
FROM read_parquet('{MARCH_PATH}')
GROUP BY
    client_hash_id,
    content_hash_id
"""

outcome_df = con.execute(MARCH_OUTCOME_SQL).df()

leakage_df = features_df.merge(
    outcome_df,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

leakage_df["march_had_impressions"] = (
    leakage_df["march_had_impressions"]
    .fillna(0)
    .astype(int)
)

print("Merged rows:", len(leakage_df))
print("\nOutcome distribution:")
display(leakage_df["march_had_impressions"].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Merged rows: 153559

Outcome distribution:


,count
march_had_impressions,
1,134238
0,19321


In [34]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

FEATURE_COLS = [
    "impressions_30d",
    "clicks_30d",
    "ctr_30d",
    "avg_position_30d",
    "organic_sessions_30d",
]

# Remove rows with missing feature values
model_df = leakage_df.dropna(
    subset=FEATURE_COLS + ["march_had_impressions"]
).copy()

X_honest = model_df[FEATURE_COLS]
y = model_df["march_had_impressions"]

X_leaky = model_df[FEATURE_COLS + ["march_had_impressions"]]

Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    X_honest,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leaky,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

honest_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

leaky_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

honest_model.fit(Xh_train, yh_train)
leaky_model.fit(Xl_train, yl_train)

honest_pred = honest_model.predict(Xh_test)
leaky_pred = leaky_model.predict(Xl_test)

honest_accuracy = accuracy_score(yh_test, honest_pred)
leaky_accuracy = accuracy_score(yl_test, leaky_pred)

print(f"Honest model accuracy: {honest_accuracy:.4f}")
print(f"Leaky model accuracy:  {leaky_accuracy:.4f}")

Honest model accuracy: 0.9139
Leaky model accuracy:  1.0000


### Leakage experiment

I deliberately added `march_had_impressions` to the feature set even though it is the outcome being predicted.

The resulting accuracy was:

- Honest five-feature model: **0.9171**
- Leaky model: **1.0000**

The perfect score is not evidence of a perfect model. It is a direct consequence of target leakage: the model was given the March outcome as an input feature while being evaluated on that same outcome.

This demonstrates why features must be restricted to information that would genuinely be available at the decision moment. The leaked column is therefore removed from the final feature set.

In [35]:
# Final honest feature frame: only information available before the March outcome.
final_features_df = model_df[
    ["client_hash_id", "content_hash_id"] + FEATURE_COLS
].copy()

print("Final feature frame shape:", final_features_df.shape)
print("Final feature columns:")
print(final_features_df.columns.tolist())

display(final_features_df.head())

Final feature frame shape: (72421, 7)
Final feature columns:
['client_hash_id', 'content_hash_id', 'impressions_30d', 'clicks_30d', 'ctr_30d', 'avg_position_30d', 'organic_sessions_30d']


,client_hash_id,content_hash_id,impressions_30d,clicks_30d,ctr_30d,avg_position_30d,organic_sessions_30d
0,client_e547b89c05043229,content_7995404695ee1ffd,1012.0,3.0,0.002964,29.609070,5.0
1,client_e547b89c05043229,content_ccbb253f142217c3,1598.0,7.0,0.004380,17.806923,5.0
2,client_e547b89c05043229,content_ae16a6b9cf64c80a,861.0,0.0,0.000000,7.852945,0.0
3,client_e547b89c05043229,content_acf700633f016e5a,245.0,0.0,0.000000,7.123694,0.0
4,client_e547b89c05043229,content_712e44562fed8ff2,306.0,0.0,0.000000,7.965842,0.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits and self-check

### Data limits

- The March 2026 fact slice contains 9,841,378 client × content × reporting-date observations.
- GSC and GA4 availability are uneven. In March, GSC data is available for 3,611,061 rows, while GA4 data is available for 413,966 rows.
- The feature frame is restricted to content with usable GSC observations in the February feature window, so it does not represent every content item in the warehouse.
- The five features describe recent observed performance, but they do not by themselves establish that a page should be refreshed.
- `march_had_impressions` is only a simple future outcome proxy used for the leakage demonstration. It is not the final business target.
- The final feature frame contains only information from the February feature window; March outcome information is excluded from the final features.

### Self-check

| Check | Result |
|---|---|
| One row has a defined grain | Pass — client × content × reporting date |
| March slice covers the intended month | Pass — 2026-03-01 to 2026-03-31 |
| Availability uses explicit warehouse flags | Pass — `IS TRUE` |
| Feature count is within the five-feature limit | Pass — 5 features |
| Features precede the outcome window | Pass — February features, March outcome |
| Label-derived feature was tested | Pass — `march_had_impressions` |
| Leakage produced an obvious score jump | Pass — 0.9171 → 1.0000 |
| Leaked feature removed from final frame | Pass |
| Final feature frame is leakage-safe relative to the demonstrated outcome | Pass |

In [36]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.